# Correct a row-summed spectrum

This example reads closed-shutter and open-shutter polystyrene Zarr stores, fits on every polystyrene spectrum except frame 42, and corrects that held-out frame. If no cosmic ray is accepted, the spectrum is returned unchanged.

In [ ]:
import matplotlib.pyplot as plt

from cosmic_ray_bayes import SpectrumCalibrator, read_spectrum_zarr

In [ ]:
spectrum_stores = {
    "dark": "../data/spectrum/closed_shutter_1000ms_fvb.zarr",
    "polystyrene": "../data/spectrum/open_shutter_polystyrene_1000ms_fvb.zarr",
}
spectra = read_spectrum_zarr(spectrum_stores)

dark_frames = spectra.xs(("dark", 1000.0), level=("experiment", "exposure_ms"))
polystyrene_frames = spectra.xs(
    ("polystyrene", 1000.0), level=("experiment", "exposure_ms")
)
reference_frames = polystyrene_frames.drop(index=42)
raw_spectrum = polystyrene_frames.loc[42]
detector_columns = raw_spectrum.index.to_numpy()

In [ ]:
calibrator = SpectrumCalibrator.fit(dark_frames, reference_frames, exposure_ms=1000)
corrected_spectrum = calibrator.correct(raw_spectrum)

In [ ]:
measured_spectrum = raw_spectrum.to_numpy()
fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True, constrained_layout=True)
axes[0].plot(detector_columns, measured_spectrum, label="measured spectrum")
axes[0].plot(detector_columns, corrected_spectrum, label="corrected spectrum")
axes[0].set_ylabel("ADU")
axes[0].legend()
axes[1].plot(detector_columns, measured_spectrum - corrected_spectrum, color="tab:red")
axes[1].set_xlabel("detector column")
axes[1].set_ylabel("removed ADU")
plt.show()

`corrected_spectrum` has the same raw scale and detector layout as the input; only accepted cosmic-ray events can change.